# PLM Cliff Benchmark - Embed External PLMs for Homology Cliff Extension

**Author:** Santiago Maniches (ORCID 0009-0005-6480-1987), TOPOLOGICA LLC

**Instructions:**
1. Colab > Runtime > Change runtime type > T4 GPU (free) or A100 (Pro).
2. Upload `data/sequences/proteins_25k_sequences.json` from the homology-cliff repo to this notebook's `/content/`. (The file is also called `experiment2_proteins_25k_filtered.json` in the SHA256-locked pre-registration; the loader accepts either name.)
3. Run all cells. Each model takes 25-90 min depending on GPU.
4. Download the `.npy` files from `/content/` to your local `data/embeddings/` directory.
5. Rerun `code/harnesses/run_cliff.py` with the new scale name added to `SCALES` to regenerate the cliff factorial for each PLM.


In [ ]:
!pip install -q transformers torch sentencepiece
import os, torch, numpy as np, json
from transformers import T5Tokenizer, T5EncoderModel, EsmTokenizer, EsmModel, AutoTokenizer, AutoModel
device='cuda'
_candidates = [
    '/content/proteins_25k_sequences.json',
    '/content/experiment2_proteins_25k_filtered.json',
]
_path = next((p for p in _candidates if os.path.exists(p)), _candidates[0])
with open(_path) as f: doc=json.load(f)
seqs=[e['sequence'] for e in doc['test_set']]
print(f'{len(seqs)} sequences from {_path}, max_len {max(len(s) for s in seqs)}')


In [ ]:
# ProtT5-XL-UniRef50 (dim 1024)
tok=T5Tokenizer.from_pretrained('Rostlab/prot_t5_xl_uniref50', do_lower_case=False)
mdl=T5EncoderModel.from_pretrained('Rostlab/prot_t5_xl_uniref50').to(device).eval()
embs=[]
for i in range(0,len(seqs),4):
    b=[' '.join(list(s[:1022])) for s in seqs[i:i+4]]
    ids=tok(b,padding=True,return_tensors='pt').to(device)
    with torch.no_grad(): out=mdl(**ids).last_hidden_state
    m=ids['attention_mask'].unsqueeze(-1).float()
    embs.append(((out*m).sum(1)/m.sum(1)).cpu().numpy())
    if i%500==0: print(f'{i}/{len(seqs)}')
emb=np.concatenate(embs).astype(np.float32); emb/=np.linalg.norm(emb,axis=1,keepdims=True)+1e-8
np.save('/content/embeddings_prot_t5.npy',emb); print('prot_t5',emb.shape)

In [ ]:
# ESM-2 t33 650M (dim 1280) — closes the ESM-2 scaling ladder
tok=EsmTokenizer.from_pretrained('facebook/esm2_t33_650M_UR50D')
mdl=EsmModel.from_pretrained('facebook/esm2_t33_650M_UR50D').to(device).eval()
embs=[]
for i in range(0,len(seqs),2):
    b=[s[:1022] for s in seqs[i:i+2]]
    ids=tok(b,padding=True,return_tensors='pt').to(device)
    with torch.no_grad(): out=mdl(**ids).last_hidden_state
    m=ids['attention_mask'].unsqueeze(-1).float()
    embs.append(((out*m).sum(1)/m.sum(1)).cpu().numpy())
    if i%500==0: print(f'{i}/{len(seqs)}')
emb=np.concatenate(embs).astype(np.float32); emb/=np.linalg.norm(emb,axis=1,keepdims=True)+1e-8
np.save('/content/embeddings_t33.npy',emb); print('t33',emb.shape)

In [ ]:
# SaProt-650M-AF2 (sequence-only mode)
tok=AutoTokenizer.from_pretrained('westlake-repl/SaProt_650M_AF2')
mdl=AutoModel.from_pretrained('westlake-repl/SaProt_650M_AF2',trust_remote_code=True).to(device).eval()
embs=[]
for i in range(0,len(seqs),2):
    b=[''.join(a+'#' for a in s[:1022]) for s in seqs[i:i+2]]
    ids=tok(b,padding=True,truncation=True,max_length=2044,return_tensors='pt').to(device)
    with torch.no_grad(): out=mdl(**ids).last_hidden_state
    m=ids['attention_mask'].unsqueeze(-1).float()
    embs.append(((out*m).sum(1)/m.sum(1)).cpu().numpy())
    if i%500==0: print(f'{i}/{len(seqs)}')
emb=np.concatenate(embs).astype(np.float32); emb/=np.linalg.norm(emb,axis=1,keepdims=True)+1e-8
np.save('/content/embeddings_saprot.npy',emb); print('saprot',emb.shape)

## Download outputs

Save under the public-release embedding directory layout:

- `/content/embeddings_prot_t5.npy` -> local `data/embeddings/embeddings_prot_t5.npy`
- `/content/embeddings_t33.npy`     -> local `data/embeddings/embeddings_t33.npy`
- `/content/embeddings_saprot.npy`  -> local `data/embeddings/embeddings_saprot.npy`

Then add the scale name(s) to `SCALES` in `code/harnesses/run_cliff.py` and re-run the factorial for each.
